# 01 Data Preparation 
### What is done here: 
 - Download must be done manually as Kaggle requires authentification.
 - Raw Data from Wikidata is preprocessed to extract the relations
### Data Source: https://www.kaggle.com/datasets/kenshoresearch/kensho-derived-wikimedia-data?select=statements.csv

In [1]:
import pandas as pd

In [ ]:
# Read necessary DataFrames:
node_df = pd.read_parquet('../03_data/preprocessed_data/nodes_dbpedia.parquet')
rels = pd.read_csv('../03_data/raw_data/wikidata/statements.csv')
page_df = pd.read_csv("../03_data/raw_data/wikidata/page.csv")
mapping = pd.read_parquet('../03_data/preprocessed_data/mapping_dbpedia_wikidata.parquet')

In [25]:
# Map the source_item_id to source_page_id because mapping contains page_id only
left = rels.merge(page_df[["page_id","item_id"]], left_on="source_item_id", right_on="item_id", how="inner")
left.loc[:,"source_page_id"]= left.loc[:,"page_id"]
both = left[["source_page_id","target_item_id"]].merge(page_df[["page_id","item_id"]], 
                                                       left_on="target_item_id", 
                                                       right_on="item_id", 
                                                       how="inner")
both.loc[:,"target_page_id"]= both.loc[:,"page_id"]
wikidata_relations = both[both["source_page_id"]!=both["target_page_id"]][["source_page_id","target_page_id"]]
mapping_df = node_df.merge(mapping, left_on="id", right_on="id",how="inner")

In [27]:
mapping_df = node_df.merge(mapping, left_on="id", right_on="id",how="inner")

In [29]:
wikidata_relations

,source_page_id,target_page_id
0,31880,50723558
1,31880,6207679
2,31880,17365007
3,31880,435544
4,31880,33196876
...,...,...
27714948,2368655,105200
27714949,2368655,962148
27714950,62057360,528282
27714951,62276323,682482


In [ ]:
map_wikidata_rels = mapping_df["page_id"].to_dict()
inv_map_wikidata_rels = {v: k for k, v in map_wikidata_rels.items()}

wikidata_relations["source_index"] = wikidata_relations["source_page_id"].map(inv_map_wikidata_rels)#.astype(int)
wikidata_relations["target_index"] = wikidata_relations["target_page_id"].map(inv_map_wikidata_rels)#.astype(int)
# Drop all Rows that contain any NaN field -> This are edges that do not belong to two nodes of the target knowledge graph
wikidata_relations.dropna(inplace=True)
wikidata_relations["source_index"] = wikidata_relations["source_index"].astype(int)
wikidata_relations["target_index"] = wikidata_relations["target_index"].astype(int)

In [33]:
wikidata_relations

,source_page_id,target_page_id,source_index,target_index
1,31880,6207679,14513,1067696
2,31880,17365007,14513,2064940
3,31880,435544,14513,167975
4,31880,33196876,14513,3434732
5,31880,41157583,14513,4095865
...,...,...,...,...
27714945,15534771,52713,1914504,24529
27714946,15534771,31772,1914504,14461
27714947,2368655,52713,555569,24529
27714948,2368655,105200,555569,45737


In [34]:
mapping_df.to_parquet("../03_data/preprocessed_data/dbpedia_wikidata_nodes.parquet")
wikidata_relations[["source_index","target_index"]].to_parquet("../03_data/preprocessed_data/wikidata_relations_networkit.parquet")

In [35]:
len(node_df)

4641780